# sprime demonstration: five routes to S'

**Tailor this notebook** to your audience. It is the primary hands-on walkthrough (load/process, Delta S', optional CSV export).

## Setup

1. Create / activate a venv with sprime installed: `pip install -e .` from the repo root.
2. Open this notebook from **anywhere inside the clone** (repo root, `docs/usage`, etc.) — the next cell **walks upward** to find `src/sprime` and `docs/usage`. If that fails, set **`REPO_ROOT_MANUAL`** in the code cell.

### Import-time choices (required)

Every **`SPrime.load`** / **`RawDataset.load_from_file`** must declare:

- **`skip_control_response_normalization`**: `False` = library will use **`Control_Response`** (DMSO/vehicle) when **processing** raw curves; `True` = responses are already on the post-control scale (empty `Control_Response` allowed on those rows).
- **`response_normalization`**: `"asymptote_normalized"` (ratio -> max-normalize -> x100) or `"response_scale"` (ratio -> x100 only). This must match **how the assay sheet / ETL defined** the curve, not an afterthought.

When **`skip_control_response_normalization=True`**, sprime does **not** re-run the ratio/x100 pipeline at process time; **`response_normalization`** still **documents** the intended interpretation of **`Responses`** for your records and for consistency with `fit_hill_from_raw_data`.

**Canonical narrative:** [`../background/s_prime_derivation_pipeline.md`](../background/s_prime_derivation_pipeline.md).

The next section stubs **all five** common routes end-to-end.

In [38]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# If auto-detect fails, set this to your clone, e.g. Path(r"C:\Users\you\Projects\sprime")
REPO_ROOT_MANUAL = None


def find_sprime_repo_root(start=None):
    """Walk upward from cwd until we find src/sprime + docs/usage (sprime clone layout)."""
    here = (start or Path.cwd()).resolve()
    for d in (here, *here.parents):
        src_sprime = d / "src" / "sprime"
        usage = d / "docs" / "usage"
        if src_sprime.is_dir() and (src_sprime / "__init__.py").is_file() and usage.is_dir():
            return d
    raise FileNotFoundError(
        "Could not find sprime repo root (expected src/sprime and docs/usage). "
        "Open the notebook from inside the clone or set REPO_ROOT_MANUAL above."
    )


REPO_ROOT = Path(REPO_ROOT_MANUAL).resolve() if REPO_ROOT_MANUAL else find_sprime_repo_root()
USAGE = REPO_ROOT / "docs" / "usage"

%cd {REPO_ROOT}



_src = REPO_ROOT / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from sprime import (  # noqa: E402 -- ignore the lint warning on import placement
    HillCurveParams,
    ScreeningDataset,
    SPrime,
    calculate_s_prime_from_params,
)

print(f"REPO_ROOT={REPO_ROOT}")
print(f"USAGE={USAGE}")

try:
    print("sprime version:", version("sprime"))
except PackageNotFoundError:
    print("sprime is not installed")

C:\Users\enact\Projects\sprime
REPO_ROOT=C:\Users\enact\Projects\sprime
USAGE=C:\Users\enact\Projects\sprime\docs\usage
sprime version: 0.1.1.dev0+g295330320.d20260312


## Five routes to S'

| # | DMSO step at process? | Normalization after ratio? | Typical source |
|---|------------------------|-----------------------------|----------------|
| **1** | Yes (`Control_Response`) | Yes (`asymptote_normalized`) | Raw plate readouts + vehicle row; **recommended** default for strict CSVs like `demo_raw_vehicle_control_*.csv`. |
| **2** | No (`skip_...=True`) | Declared `asymptote_normalized` | Responses **already** on the asymptote-normalized x100 scale upstream (e.g. ipNF-style **`demo_precontrol_normalized_*.csv`**). |
| **3** | No | Declared `response_scale` | **Zamora et al.-style** workflows where curves were built from **ratio x100 without** max-normalize; **we still recommend asymptote normalization** when you can align to a reference method. |
| **4** | Yes | `response_scale` | Raw readouts + `Control_Response`, but **only** ratio then x100 (sheet *Nonnormalized* path). **Not recommended** unless you must reproduce a legacy column exactly. |
| **5** | N/A (no raw fit) | N/A | **Pre-calculated Hill parameters** (or external fit) -> S' via `calculate_s_prime_from_params` or precalc-only CSV. |

### Raw vehicle demo: list vs columns (same curve)

For **strict DMSO + raw** demos aligned with `tests/fixtures/SPrime_variation_reference.csv`:

- **List layout:** [`demo_raw_vehicle_control_raw_list.csv`](demo_raw_vehicle_control_raw_list.csv) &mdash; `Responses` / `Concentrations` comma-separated; `values_as="list"`.
- **Columns layout:** [`demo_raw_vehicle_control_s_prime.csv`](demo_raw_vehicle_control_s_prime.csv) &mdash; `DATA0`..`DATA7` / `CONC0`..`CONC7`; `values_as="columns"` (default).

Both rows carry the same biology and **`Control_Response`** (35.3). **[`template_raw.csv`](template_raw.csv)** is a generic toy example, not this fixture.

### Route 2: decimals, not percents

If your upstream table expressed **response as % of DMSO control**, convert to a **dimensionless decimal ratio** before putting values in **`Responses`** (e.g. divide by 100, or store `1.15` not `115`). sprime expects **plain floats** in the same units your Hill model used when those numbers were produced.

### Route 5: strong caution

If you import **EC50 / ZERO / INF** (or Upper/Lower) from a file or another tool, **your S' only matches biology if those parameters used the same DMSO, normalization, and curve convention** as this assay. **Within one assay** mixing conventions may cancel partially; **across instruments, templates, or publications**, mismatches **amplify** and S' magnitudes become incomparable. **Check the original assay design** for how zero asymptote, inf asymptote, and EC50 were defined. **We recommend starting from raw dose-response data** when you have it, so `skip_control_response_normalization` and `response_normalization` match the documented pipeline.

In [26]:
# =============================================================================
# 1) DMSO (vehicle) ratio applied at process + asymptote-normalized (RECOMMENDED)
#    Raw plate units; Control_Response = vehicle readout (e.g. 35.3).
#    List layout (below) OR columns: demo_raw_vehicle_control_s_prime.csv + values_as="columns".
# =============================================================================
p1 = USAGE / "demo_raw_vehicle_control_raw_list.csv"
raw1, _ = SPrime.load(
    p1,
    values_as="list",
    skip_control_response_normalization=False,
    response_normalization="asymptote_normalized",
)
# Same curve, Path A columns (DATA*/CONC*):
# p1_cols = USAGE / "demo_raw_vehicle_control_s_prime.csv"
# raw1, _ = SPrime.load(
#     p1_cols,
#     values_as="columns",
#     skip_control_response_normalization=False,
#     response_normalization="asymptote_normalized",
# )
scr1, _ = SPrime.process(raw1)
print("Route 1 S':", [(x.cell_line.name, x.s_prime) for x in scr1.profiles])

_demo_out = REPO_ROOT / "docs" / "usage" / "_notebook_output"
_demo_out.mkdir(parents=True, exist_ok=True)
scr1.export_to_csv(_demo_out / "s_prime_values.csv")
print("Wrote demonstrated CSV:", _demo_out / "s_prime_values.csv")


DATA PROCESSING SUMMARY
Total Rows:                 2
Rows Processed:             1
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            1
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 1
Profiles Failed:             0

No warnings found.

Route 1 S': [('NF2-/-', 6.867631094104203)]
Wrote demonstrated CSV: C:\Users\enact\Projects\sprime\docs\usage\_notebook_output\s_prime_values.csv


In [27]:
# =============================================================================
# 2) No DMSO step at process + asymptote-normalized (declare intent)
#    Responses already on final analysis scale (upstream applied ratio + max-norm + x100).
#    If your source used "% of control", convert to decimal ratio before CSV (see markdown).
# =============================================================================
p2 = USAGE / "demo_precontrol_normalized_raw_list.csv"
raw2, _ = SPrime.load(
    p2,
    values_as="list",
    skip_control_response_normalization=True,
    response_normalization="asymptote_normalized",
)
scr2, _ = SPrime.process(raw2, allow_overwrite_precalc_params=True)
print("Route 2 S':", [(x.cell_line.name, x.s_prime) for x in scr2.profiles])


DATA PROCESSING SUMMARY
Total Rows:                 2
Rows Processed:             1
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            1
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 1
Profiles Failed:             0

No warnings found.

Route 2 S': [('ipNF96.11C', -18.84880324391443)]


In [28]:
# =============================================================================
# 3) No DMSO at process + response-scale only (Zamora et al.-style; not our default)
#    Responses already = (test/DMSO) * 100 without max-normalization step.
#    Set YOUR_CSV_PATH to your file, then uncomment.
# =============================================================================
# YOUR_CSV_PATH = USAGE / "your_file.csv"
p3_cols = USAGE / "demo_raw_vehicle_control_s_prime.csv"
raw3, _ = SPrime.load(
    p3_cols,
    values_as="columns",  # or "list"
    skip_control_response_normalization=True,
    response_normalization="response_scale",
)
scr3, _ = SPrime.process(raw3)
print("Route 3 S':", [(x.cell_line.name, x.s_prime) for x in scr3.profiles])


DATA PROCESSING SUMMARY
Total Rows:                 2
Rows Processed:             1
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            1
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 1
Profiles Failed:             0

No warnings found.

Route 3 S': [('NF2-/-', -5.981874833041447)]


In [29]:

# =============================================================================
# 4) DMSO at process + response-scale only (legacy / non-normalized sheet column)
#    NOT RECOMMENDED unless reproducing a specific published pipeline.
# =============================================================================
p4_cols = USAGE / "demo_raw_vehicle_control_s_prime.csv"

# Or same curve with columns:
raw4, _ = SPrime.load(
    p4_cols,
    values_as="columns",
    skip_control_response_normalization=False,
    response_normalization="response_scale",
)
scr4, _ = SPrime.process(raw4)
print("Route 4 S':", [(x.cell_line.name, x.s_prime) for x in scr4.profiles])



DATA PROCESSING SUMMARY
Total Rows:                 2
Rows Processed:             1
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            1
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 1
Profiles Failed:             0

No warnings found.

Route 4 S': [('NF2-/-', 7.02315650025724)]


In [30]:
# =============================================================================
# 5) S' from Hill parameters only (precalc / external fit)
#    Prior work MUST match DMSO + normalization assumptions or S' magnitudes skew.
#    Prefer raw re-fit (routes 1-4) when possible.
# =============================================================================
p5 = USAGE / "demo_precontrol_normalized_precalc.csv"
raw5, _ = SPrime.load(p5, response_normalization="asymptote_normalized")
scr5, _ = SPrime.process(raw5)
print("Route 5 (precalc CSV) S':", [(x.cell_line.name, x.s_prime) for x in scr5.profiles])




DATA PROCESSING SUMMARY
Total Rows:                 4
Rows Processed:             3
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            3
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 3
Profiles Failed:             0

WARNINGS: 3 total
  PRE_CALC_ONLY: 3
    N/A: Using pre-calc Hill parameters as-is for 'Trifluop (ID:NCGC00013226-15, CL:ipNF96.11C)
    N/A: Using pre-calc Hill parameters as-is for 'Trifluop (ID:NCGC00013226-15, CL:ipnf05.5 mc)
    N/A: Using pre-calc Hill parameters as-is for 'Trifluop (ID:NCGC00013226-15, CL:ipnf02.3)

Route 5 (precalc CSV) S': [('ipNF96.11C', -5.044950504499101), ('ipnf05.5 mc', -5.046821041487659), ('ipnf02.3', -5.032656731182978)]


In [31]:
# =============================================================================
# 6) S' from Hill parameters only (precalc / external fit)
#    Prior work MUST match DMSO + normalization assumptions or S' magnitudes skew.
#    Prefer raw re-fit (routes 1-4) when possible.
# Or direct formula (same definition as library uses from fitted params):
hp = HillCurveParams(ec50=0.2, zero_asymptote=100.0, inf_asymptote=3.0, steepness_coefficient=-1.0)
sp_direct = calculate_s_prime_from_params(hp.ec50, hp.zero_asymptote, hp.inf_asymptote)
print("Route 5 (manual params) S':", sp_direct)

Route 5 (manual params) S': 6.877297134307935


In [32]:
# =============================================================================
# 7) In-memory raw arrays -> fit_hill_from_raw_data -> S' (no CSV)
#    Same pipeline as route 1 when you already have Python lists.
#    Prior work MUST match DMSO + normalization assumptions or S' magnitudes skew.
# =============================================================================
from sprime import calculate_s_prime_from_params, fit_hill_from_raw_data

concentrations_uM = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
raw_response_noncontrolled = [41.24, 40.86, 38.78, 38.38, 6.64, 0.94, 0.0, 0.0]
dmso_control = 35.3


# ----- Arrays -> 4PL fit -> S' (recommended when you have raw points in memory) -----

hill = fit_hill_from_raw_data(
    raw_response_noncontrolled,
    concentrations_uM,
    control_response=dmso_control,
    skip_control_response_normalization=False,
    response_normalization="asymptote_normalized",
    concentration_units="microM",
)
s_prime_from_fit = calculate_s_prime_from_params(
    hill.ec50, hill.zero_asymptote, hill.inf_asymptote
)
print("S':", s_prime_from_fit)


S': 6.867631094104203


## Delta S' — two CSV paths (explicit)

1. **`demo_raw_vehicle_control_delta.csv`**: raw `DATA*` / `CONC*` plus **`Control_Response`** (vehicle/DMSO). **S' is derived** by `SPrime.process` (Hill fit, then S'). Then **`calculate_delta_s_prime`** for **NF2-/- Ref** vs **NF2-/- Test**.

2. **`demo_precontrol_normalized_precalc.csv`**: **Hill parameters and an optional `S'` column** (`AC50`, `Upper`/`Lower`, `S'`); no dose–response points. **`SPrime.load`** still requires **`response_normalization=`** (library contract + metadata); it does **not** re-apply pipelines to precalc rows—coefficients must already match the assay normalization used when they were produced. **`process`** recomputes S' from those coefficients (sheet `S'` is informational). **`PRE_CALC_ONLY`** in the report is expected. Then compare **ipnf05.5 mc** (reference) vs **ipNF96.11C** (test).

For **`demo_precontrol_normalized_delta.csv`** (two raw precontrol rows, same compound / two cell lines), see **Optional: CSV export** below after you compute S' and Delta S'.

In [33]:
# Delta S' example A: raw vehicle + two cell lines (S' derived from dose-response fit)
p_delta_vehicle = USAGE / "demo_raw_vehicle_control_delta.csv"
raw_dv, _ = SPrime.load(
    p_delta_vehicle,
    values_as="columns",
    skip_control_response_normalization=False,
    response_normalization="asymptote_normalized",
)
screen_dv, _ = SPrime.process(raw_dv)
print("S' (derived):", [(p.cell_line.name, p.s_prime) for p in screen_dv.profiles])
delta_dv = screen_dv.calculate_delta_s_prime(
    reference_cell_lines="NF2-/- Ref",
    test_cell_lines=["NF2-/- Test_1", "NF2-/- Test_2"],
)
for ref_cl, rows in delta_dv.items():
    for r in rows:
        print(
            f"  {ref_cl} vs {r['test_cell_line']}: Delta S' = {r['delta_s_prime']:.6f} "
            f"(S'_ref={r['s_prime_ref']:.6f}, S'_test={r['s_prime_test']:.6f})"
        )

_demo_out = REPO_ROOT / "docs" / "usage" / "_notebook_output"
_demo_out.mkdir(parents=True, exist_ok=True)
screen_dv.export_to_csv(_demo_out / "delta_s_prime_example_a_profiles.csv")
ScreeningDataset.export_delta_s_prime_to_csv(
    delta_dv, _demo_out / "delta_s_prime_example_a_delta.csv"
)
print(
    "Wrote demonstrated CSVs:",
    _demo_out / "delta_s_prime_example_a_profiles.csv",
    _demo_out / "delta_s_prime_example_a_delta.csv",
)



DATA PROCESSING SUMMARY
Total Rows:                 4
Rows Processed:             3
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            3
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 3
Profiles Failed:             0

No warnings found.

S' (derived): [('NF2-/- Ref', -3.1764092497421257), ('NF2-/- Test_1', -6.143367466705312), ('NF2-/- Test_2', -4.720064185865709)]
  NF2-/- Ref vs NF2-/- Test_1: Delta S' = 2.966958 (S'_ref=-3.176409, S'_test=-6.143367)
  NF2-/- Ref vs NF2-/- Test_2: Delta S' = 1.543655 (S'_ref=-3.176409, S'_test=-4.720064)
Wrote demonstrated CSVs: C:\Users\enact\Projects\sprime\docs\usage\_notebook_output\delta_s_prime_example_a_profiles.csv C:\Users\enact\Projects\sprime\docs\usage\_notebo

In [34]:
# Delta S' example B: precalc-only CSV (Hill coefficients + optional S' column; no dose-response points)
# `response_normalization` is required on every SPrime.load (validation + recorded intent). It does NOT
# re-apply ratio/max-norm to precalc rows—your AC50/Upper/Lower must already reflect whatever pipeline
# produced them (e.g. asymptote-normalized x100 vs response-scale x100). Match the lab's convention here.
precalculated_hill_coefficients_file = USAGE / "demo_precontrol_normalized_precalc.csv"
precalc_dataset, _ = SPrime.load(
    precalculated_hill_coefficients_file,
    response_normalization="asymptote_normalized",
)
screening_precalc, _ = SPrime.process(precalc_dataset)
print("S' (recomputed from Hill params):", [(p.cell_line.name, p.s_prime) for p in screening_precalc.profiles])
delta_precalc = screening_precalc.calculate_delta_s_prime(
    reference_cell_lines="ipnf05.5 mc",
    test_cell_lines="ipNF96.11C",
)
for ref_cl, rows in delta_precalc.items():
    for r in rows:
        print(
            f"  {ref_cl} vs {r['test_cell_line']}: Delta S' = {r['delta_s_prime']:.6f} "
            f"(S'_ref={r['s_prime_ref']:.6f}, S'_test={r['s_prime_test']:.6f})"
        )

_demo_out = REPO_ROOT / "docs" / "usage" / "_notebook_output"
_demo_out.mkdir(parents=True, exist_ok=True)
ScreeningDataset.export_delta_s_prime_to_csv(
    delta_precalc, _demo_out / "delta_s_prime_from_precalc.csv"
)
print("Wrote demonstrated CSV:", _demo_out / "delta_s_prime_from_precalc.csv")



DATA PROCESSING SUMMARY
Total Rows:                 4
Rows Processed:             3
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            3
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 3
Profiles Failed:             0

WARNINGS: 3 total
  PRE_CALC_ONLY: 3
    N/A: Using pre-calc Hill parameters as-is for 'Trifluop (ID:NCGC00013226-15, CL:ipNF96.11C)
    N/A: Using pre-calc Hill parameters as-is for 'Trifluop (ID:NCGC00013226-15, CL:ipnf05.5 mc)
    N/A: Using pre-calc Hill parameters as-is for 'Trifluop (ID:NCGC00013226-15, CL:ipnf02.3)

S' (recomputed from Hill params): [('ipNF96.11C', -5.044950504499101), ('ipnf05.5 mc', -5.046821041487659), ('ipnf02.3', -5.032656731182978)]
  ipnf05.5 mc vs ipNF96.11C: De

### Four `load` + `process` combinations (Delta S' context)

| | **`allow_overwrite_precalc_params=False`** (default) | **`True`** |
|---|------------------------------------------------------|------------|
| **`skip_control_response_normalization=False`** (strict `Control_Response` on raw rows) | **A** — mixed raw+precalc on the *same* row **raises** | **B** — fit from raw overwrites precalc (warning) |
| **`skip_control_response_normalization=True`** (empty `Control_Response` allowed; pre-control-normalized) | **C** — same as A for conflicts | **D** — same as B for conflicts |

**Demo pairing:** **A/B** use **`demo_raw_vehicle_control_delta.csv`** (vehicle + raw curves). **C/D** use **`demo_precontrol_normalized_delta.csv`** (empty control + raw curves on analysis scale). These files do **not** mix raw+precalc on one line, so **A vs B** and **C vs D** give the **same** screening output here; the flags still show the call pattern for sheets that *do* contain both.

`response_normalization` is required on every `load` (use the same value for A-D below).

In [35]:
# A-D: same Delta S' comparison targets as examples A (vehicle) and scenario 3 / C-D (precontrol).
RN = "asymptote_normalized"
p_vehicle_delta = USAGE / "demo_raw_vehicle_control_delta.csv"
p_precontrol_delta = USAGE / "demo_precontrol_normalized_delta.csv"


def _print_delta(label, screening, ref, test):
    d = screening.calculate_delta_s_prime(
        reference_cell_lines=ref,
        test_cell_lines=test,
    )
    print(f"\n--- {label} ---")
    for ref_cl, rows in d.items():
        for r in rows:
            print(
                f"  {ref_cl} vs {r['test_cell_line']}: Delta S' = {r['delta_s_prime']:.6f} "
                f"(S'_ref={r['s_prime_ref']:.6f}, S'_test={r['s_prime_test']:.6f})"
            )


# A: skip=False, allow_overwrite_precalc_params=False
raw_a, _ = SPrime.load(
    p_vehicle_delta,
    values_as="columns",
    skip_control_response_normalization=False,
    response_normalization=RN,
)
scr_a, _ = SPrime.process(raw_a, allow_overwrite_precalc_params=False)
_print_delta(
    "A  skip_control_response_normalization=False, allow_overwrite_precalc_params=False",
    scr_a,
    "NF2-/- Ref",
    ["NF2-/- Test_1", "NF2-/- Test_2"],
)

# B: skip=False, allow_overwrite_precalc_params=True (identical to A for these demo rows)
raw_b, _ = SPrime.load(
    p_vehicle_delta,
    values_as="columns",
    skip_control_response_normalization=False,
    response_normalization=RN,
)
scr_b, _ = SPrime.process(raw_b, allow_overwrite_precalc_params=True)
_print_delta(
    "B  skip_control_response_normalization=False, allow_overwrite_precalc_params=True",
    scr_b,
    "NF2-/- Ref",
    ["NF2-/- Test_1", "NF2-/- Test_2"],
)

# C: skip=True, allow_overwrite_precalc_params=False
raw_c, _ = SPrime.load(
    p_precontrol_delta,
    values_as="columns",
    skip_control_response_normalization=True,
    response_normalization=RN,
)
scr_c, _ = SPrime.process(raw_c, allow_overwrite_precalc_params=False)
_print_delta(
    "C  skip_control_response_normalization=True, allow_overwrite_precalc_params=False",
    scr_c,
    "ipnf05.5 mc",
    "ipNF96.11C",
)

# D: skip=True, allow_overwrite_precalc_params=True (identical to C for these demo rows)
raw_d, _ = SPrime.load(
    p_precontrol_delta,
    values_as="columns",
    skip_control_response_normalization=True,
    response_normalization=RN,
)
scr_d, _ = SPrime.process(raw_d, allow_overwrite_precalc_params=True)
_print_delta(
    "D  skip_control_response_normalization=True, allow_overwrite_precalc_params=True",
    scr_d,
    "ipnf05.5 mc",
    "ipNF96.11C",
)


DATA PROCESSING SUMMARY
Total Rows:                 4
Rows Processed:             3
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            3
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 3
Profiles Failed:             0

No warnings found.


--- A  skip_control_response_normalization=False, allow_overwrite_precalc_params=False ---
  NF2-/- Ref vs NF2-/- Test_1: Delta S' = 2.966958 (S'_ref=-3.176409, S'_test=-6.143367)
  NF2-/- Ref vs NF2-/- Test_2: Delta S' = 1.543655 (S'_ref=-3.176409, S'_test=-4.720064)

DATA PROCESSING SUMMARY
Total Rows:                 4
Rows Processed:             3
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            3
Profiles with S' Calculated: 0

ValueError: Pre-calculated curve parameters (AC50, Zero_asymptote, Inf_asymptote, Hill_Slope, r2) would be overwritten by fitted values for compound 'Trifluoperazine hydrochloride' (Compound_ID: NCGC00013226-15) in cell line 'ipNF96.11C'. Set allow_overwrite_precalc_params=True to permit.

## Optional: `response_pipeline` (manual arrays)

For **CSV workflows**, routes **1-4** above already apply **`pipeline_asymptote_normalized`** or **`pipeline_response_scale`** inside **`SPrime.process`** when you set **`skip_control_response_normalization`** and **`response_normalization`** correctly.

Use **`sprime.response_pipeline`** directly only when you are building **in-memory** response vectors (no CSV) and need the same algebra as `tests/fixtures/SPrime_variation_reference.csv`.

In [36]:
# from sprime.response_pipeline import pipeline_response_scale, pipeline_asymptote_normalized
# raw = [41.24, 40.86, 38.78, 38.38, 6.64, 0.94, 0.0, 0.0]
# ctrl = 35.3
# scaled = pipeline_response_scale(raw, ctrl)
# print("x100-style:", scaled[:3], "...")

## Optional: CSV export (precontrol delta demo)

Loads **`demo_precontrol_normalized_delta.csv`**, fits, then writes **`master_s_prime_table.csv`** and **`delta_s_prime_table.csv`** using **`ScreeningDataset` export helpers** (same idea as the old script scenario 3).

In [37]:
out = REPO_ROOT / "docs" / "usage" / "_notebook_output"
out.mkdir(parents=True, exist_ok=True)

raw_export, _ = SPrime.load(
    USAGE / "demo_precontrol_normalized_delta.csv",
    skip_control_response_normalization=True,
    response_normalization="asymptote_normalized",
)
screening_export, _ = SPrime.process(raw_export, allow_overwrite_precalc_params=True)
screening_export.export_to_csv(out / "master_s_prime_table.csv")

delta_export = screening_export.calculate_delta_s_prime(
    reference_cell_lines="ipnf05.5 mc",
    test_cell_lines="ipNF96.11C",
)
ScreeningDataset.export_delta_s_prime_to_csv(delta_export, out / "delta_s_prime_table.csv")
print("Wrote:", out / "master_s_prime_table.csv", out / "delta_s_prime_table.csv")


DATA PROCESSING SUMMARY
Total Rows:                 3
Rows Processed:             2
Rows Skipped:                0
Compounds Loaded:            1
Profiles Created:            2
Profiles with S' Calculated: 0
Profiles Failed:             0

No warnings found.


DATA PROCESSING SUMMARY
Total Rows:                 0
Rows Processed:             0
Rows Skipped:                0
Compounds Loaded:            0
Profiles Created:            0
Profiles with S' Calculated: 2
Profiles Failed:             0

WARNINGS: 1 total
  OVERWRITE_PRECALC_PARAMS: 1
    N/A: Pre-calculated curve parameters (AC50, Zero_asympt (ID:NCGC00013226-15, CL:ipNF96.11C)

Wrote: C:\Users\enact\Projects\sprime\docs\usage\_notebook_output\master_s_prime_table.csv C:\Users\enact\Projects\sprime\docs\usage\_notebook_output\delta_s_prime_table.csv
